## Usernames scraping

This code aims to scrap the r/france subreddit to collect all usernames.

### Scrap usernames

In [2]:
import requests
import time

URL = "https://api.pullpush.io/reddit/submission/search/"
headers = {
    "User-Agent": "Mozilla/5.0 (Our research; contact: placeholder@example.com)"
}

with open("usernames.txt", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()
    lines = []
    for line in raw_lines:
        lines.append(line.strip('\n'))
    last_line = lines[-1]
    try :
        last_line = float(last_line)
        before = last_line
        authors = set(lines[-2::-1])
    except:
        print("No last period saved")
        before = int(time.time())
        authors = set(lines)

unique = len(authors)
while unique < 100000:
    params = {
        "subreddit": "france",
        "size": 100,
        "before": before,
        "sort": "desc"
    }

    r = requests.get(URL,headers=headers, params=params, timeout=30)
    r.raise_for_status()
    data = r.json().get("data", [])

    if not data:
        print("No more data.")
        break

    for post in data:
        author = post.get("author")
        if author and author != "[deleted]":
            authors.add(author)
    
    unique = len(set(authors))

    # store last production date
    before = data[-1]["created_utc"]

    print(f"Authors: {len(authors)} | Unique: {unique}")

    time.sleep(1)


Authors: 4683 | Unique: 4683
Authors: 4732 | Unique: 4732
Authors: 4777 | Unique: 4777


HTTPError: 525 Server Error: <none> for url: https://api.pullpush.io/reddit/submission/search/?subreddit=france&size=100&before=1741521214.0&sort=desc

In [3]:
import requests
import time
from datetime import datetime, timezone 

def safe_request(url, headers, params, retries=5, backoff=5):
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=30)
            r.raise_for_status()
            return r

        except requests.exceptions.SSLError as e:
            print(
                f"SSL error (Cloudflare 525 likely) "
                f"(attempt {attempt}/{retries}): {e}"
            )
            time.sleep(backoff * attempt)

        except requests.exceptions.HTTPError as e:
            status = e.response.status_code if e.response else None
            print(
                f"HTTP error {status} "
                f"(attempt {attempt}/{retries})"
            )
            time.sleep(backoff * attempt)

        except requests.exceptions.RequestException as e:
            print(
                f"Network error "
                f"(attempt {attempt}/{retries}): {e}"
            )
            time.sleep(backoff * attempt)

    return None


URL = "https://api.pullpush.io/reddit/submission/search/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Research project; contact: placeholder@example.com)"
}
FILENAME = "usernames3.txt"
TARGET_UNIQUE = 100000
SUBREDDIT = "france"
PAGE_SIZE = 100
SLEEP_SECONDS = 1

def load_state(filename):
    """
    File format:
    - one username per line
    - last line MUST be a Unix timestamp (int)
    """
    try:
        with open(filename, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        return set(), None
    if not lines:
        return set(), None
    try:
        before = int(lines[-1])
        authors = set(lines[:-1])
        return authors, before
    except ValueError:
        raise RuntimeError(
            "Corrupted state file: last line must be a Unix timestamp"
        )

def save_state(filename, authors, before):
    with open(filename, "w", encoding="utf-8") as f:
        for author in sorted(authors):
            f.write(author + "\n")
        f.write(str(int(before)) + "\n")
 
# ---- INITIAL TIMESTAMP (31 Dec 2025, 23:59 UTC) ----

start_datetime = datetime(2025, 12, 31, 23, 59, tzinfo=timezone.utc)

DEFAULT_BEFORE = int(start_datetime.timestamp())

authors, before = load_state(FILENAME)

if before is None:
    before = DEFAULT_BEFORE
    print("Starting fresh from 2025-12-31 23:59 UTC")

else:
    print(f"Resuming crawl from timestamp {before}")

while len(authors) < TARGET_UNIQUE:
    params = {
        "subreddit": SUBREDDIT,
        "size": PAGE_SIZE,
        "before": before,
        "sort": "desc"
    }

    r = safe_request(URL, HEADERS, params)
    data = r.json().get("data", [])

    if not data:
        print("No more data available.")
        break

    for post in data:
        author = post.get("author")
        if author and author != "[deleted]":
            authors.add(author)

    before = data[-1]["created_utc"]

    save_state(FILENAME, authors, before)

 

    print(f"Unique authors collected: {len(authors)}")

 

    time.sleep(SLEEP_SECONDS)

Starting fresh from 2025-12-31 23:59 UTC
Unique authors collected: 81
Unique authors collected: 159
Unique authors collected: 234
Unique authors collected: 297
Unique authors collected: 355
Unique authors collected: 414
Unique authors collected: 489
Unique authors collected: 551
Unique authors collected: 617
Unique authors collected: 671
Unique authors collected: 741
Unique authors collected: 793
Unique authors collected: 857
Unique authors collected: 906
Unique authors collected: 964
Unique authors collected: 1020
Unique authors collected: 1077
Unique authors collected: 1142
Unique authors collected: 1196
Unique authors collected: 1253
Unique authors collected: 1306
Unique authors collected: 1363
Unique authors collected: 1421
Unique authors collected: 1482
Unique authors collected: 1540
Unique authors collected: 1599
Unique authors collected: 1664
HTTP error None (attempt 1/5)
Unique authors collected: 1711
Unique authors collected: 1765
Unique authors collected: 1818
Unique authors 

### Store usernames in `usernames.txt`

In [ ]:
number = 0
with open("usernames.txt", "w", encoding="utf-8") as f:
    unique_authors = set(authors)
    for author in unique_authors:
        f.write(f"{author}\n")
    f.write(f'{before}')
